# Tutorial 7-1: Nonlinear Decision Boundaries
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 

### Beyond Straight Lines with Logistic Regression

## 1. The "Linear" Misconception
In lecture, we defined Logistic Regression as a linear model because the decision boundary is defined by the linear combination of parameters:
$$\theta^T x = 0$$

However, this "linearity" refers to the parameters $\theta$, not the features $x$. By transformng our input features—such as squaring them or creating interaction terms—we can create complex, nonlinear decision boundaries in our original feature space.

**Objective:** In this tutorial, we will:
1. Generate a "Circular" dataset that cannot be separated by a straight line.
2. Observe a standard Logistic Regression model fail to classify it.
3. Use **Polynomial Feature Mapping** to allow the model to find a circular boundary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures

# Step 1: Generate non-linear data (A circle within a circle)
X, y = make_circles(n_samples=400, factor=0.5, noise=0.1, random_state=42)

def plot_data(X, y, title):
    plt.scatter(X[y==0, 0], X[y==0, 1], color='red', marker='_', label='Class 0')
    plt.scatter(X[y==1, 0], X[y==1, 1], color='green', marker='+', label='Class 1')
    plt.title(title)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.legend()

plt.figure(figsize=(6, 6))
plot_data(X, y, "Circular Dataset (Non-linearly Separable)")
plt.show()

## 2. Standard Logistic Regression
Here we fit a standard model: $h_\theta(x) = g(\theta_0 + \theta_1 x_1 + \theta_2 x_2)$. 

Because the boundary is restricted to a line, the model will be forced to draw a line through the middle of the circles, leading to poor accuracy.

In [ ]:
model_linear = LogisticRegression()
model_linear.fit(X, y)

print(f"Linear Model Accuracy: {model_linear.score(X, y):.2%}")

## 3. Polynomial Feature Mapping
To solve this, we map our 2D features $(x_1, x_2)$ into a higher-dimensional space. By including squared terms, our hypothesis becomes:
$$h_\theta(x) = g(\theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_1^2 + \theta_4 x_2^2)$$

In this new 5D space, the decision boundary is still a "hyperplane" (linear). But when we project that hyperplane back down to our original 2D space, it appears as a **circle**.

In [ ]:
# Transform features to degree 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

model_nonlinear = LogisticRegression()
model_nonlinear.fit(X_poly, y)

print(f"Non-linear Model Accuracy: {model_nonlinear.score(X_poly, y):.2%}")

## 4. Visualizing the Decision Boundary
We will visualize the boundary by creating a mesh grid and predicting the class for every point in the grid.

In [ ]:
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

# Predict using the nonlinear model
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_poly = poly.transform(grid_points)
Z = model_nonlinear.predict(grid_poly)
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, cmap=plt.cm.RdYlGn, alpha=0.3)
plot_data(X, y, "Non-linear Decision Boundary (Circle)")
plt.show()

## Summary
* **Standard Logistic Regression** is limited to linear boundaries.
* **Feature Mapping** allows us to fit non-linear boundaries by increasing the dimensionality of our input data.
* This is the precursor to **Kernel methods**, which allow us to perform this mapping implicitly without manually calculating squared terms.